# Week 4 — Preprocessing & Feature Engineering
## Hands-on challenge: from messy data to model-ready data

You are given a synthetic survey dataset. Your goal is **not** a perfect dataset; it is a set of **defensible preprocessing decisions**.

We follow the same workflow as the lecture:

| Step | What | Before or after the split? |
|---|---|---|
| 1 | Inspect | before (read-only) |
| 2 | Remove duplicates, fix labels, flag impossible values | before — row-wise rules, nothing is learned |
| 3 | Choose target and features, check for target leakage | before |
| 4 | **Train / test split** | — |
| 5 | Missing values | after — medians/modes are learned from **train** |
| 6 | Outliers | after — investigate on **train** |
| 7 | Dates | after (row-wise, applied to both) |
| 8 | Engineer new features | after (row-wise, applied to both) |
| 9 | Encode categories | after — categories are learned from **train** |
| 10 | Scale numerical variables | after — mean/SD learned from **train** |
| 11 | Build a preprocessing pipeline | — |
| 12 | Leakage, revisited | — |

**Rule of thumb:** anything that *learns* from the data goes after the split; pure row-wise corrections can come before.

Run each worked example, inspect its output, and answer the short questions.

> ### 📘 How to use this tutorial
> - **Part A** (sections 1–12 below) is the original Week 4 notebook, now completed. Every empty cell is filled, and every question has an **Answer** cell. The **Tutorial** notes explain *why* each step is done the way it is.
> - **Part A needs `../data/week4_messy_survey.csv`**, the course file loaded in the next cell. Put it in your repo's `data/` folder if it isn't there already.
> - **Part B** (at the end) repeats the same 12-step workflow on `dataset.csv`, the student social-media dataset, so you can see the method used on new data.
>
> Run the cells top to bottom with **Run All**, or step through them with Shift + Enter.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer

df = pd.read_csv("../data/week4_messy_survey.csv")
df.head()

,respondent_id,age,gender,education,household_size,annual_income_eur,survey_date,hours_online_per_week,online_purchases_last_month,satisfaction_score,complaint_filed_after_survey
0,R0494,48.0,Male,High School,2.0,34800.0,2025-08-31,6.1,2.0,6,No
1,R0473,38.0,Non-binary,Bachelor,4.0,31000.0,2025-12-09,2.8,0.0,6,No
2,R0108,38.0,Female,Bachelor,1.0,25100.0,2025-06-15,4.3,1.0,7,No
3,R0559,26.0,Female,Bachelor,1.0,NaN,2025-10-14,4.2,0.0,9,No
4,R0536,50.0,Female,Bachelor,1.0,84800.0,2025-04-27,7.1,3.0,8,No


---
## 1. Inspect the dataset

Before changing anything, find out what you have. Inspecting the full dataset is fine — you are only *looking*, not learning parameters.

**Observe**
- How many rows and columns? What are the data types?
- Which columns contain missing values, and how many?
- Are there duplicate rows?
- What do `describe()` and `value_counts()` tell you? **Read the min and max first.**

In [ ]:
# Shape and data types
# Hint: df.shape, df.info()
print("Rows, columns:", df.shape)
df.info()

In [ ]:
# Missing values, duplicates, summary statistics
# Hint: df.isna().sum(), df.duplicated().sum(), df.describe()
print("Missing values per column:")
print(df.isna().sum().sort_values(ascending=False))

print("\nExact duplicate rows:", df.duplicated().sum())

# Read the min and max rows first: they reveal impossible values fastest.
df.describe()

In [ ]:
# Categorical counts — include missing values
# Hint: df[col].value_counts(dropna=False)
for col in ["gender", "education", "complaint_filed_after_survey"]:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False))
    print()

**Tutorial — what to look for in Section 1**

- `df.info()` shows the dtype of each column. `survey_date` is stored as text (`object`), not as a date, which is why Section 7 converts it.
- `isna().sum()` shows gaps in `annual_income_eur`, `age`, `gender`, `education` and `household_size`.
- `duplicated().sum()` is above zero, so some respondents appear twice. After removing them, 600 rows are left (Section 2).
- In `describe()`, **read the min and max first.** `age` has a maximum above 110 and `annual_income_eur` has a negative minimum. Neither is possible.
- `value_counts(dropna=False)` on `gender` and `education` shows the same category spelt several ways (`"F"`, `"female"`, `" Male"`, `"MASTER"`, …), plus `NaN`.

**Answer:** The dataset has 11 columns. It contains missing values in five columns, duplicate rows, inconsistent category labels and some impossible ages and incomes. None of this is fixed yet. Section 1 only looks.

---
## 2. Row-wise cleaning (before the split)

These steps are **rules you decide**, not statistics learned from the data. That is why they can happen before the split.

**Tasks**
1. Remove exact duplicate rows. *Why before the split?* Otherwise a copy of the same person can land in both train and test.
2. Standardise the labels in `gender` and `education` (`"F"`, `"female"`, `" Male"`, `"MASTER"`, …).
3. Replace **impossible** values with `NaN`: which ages and incomes are impossible, not just unusual?

**Think first:** which problems can be fixed automatically, and which require a decision?

In [ ]:
df_clean = df.copy().drop_duplicates()

# Standardise categorical labels with fixed, row-wise rules.
gender_map = {
    "f": "Female", "female": "Female", "m": "Male", "male": "Male",
    "non-binary": "Non-binary", "prefer not to say": "Prefer not to say",
}
education_map = {
    "high school": "High School", "bachelor": "Bachelor",
    "master": "Master", "phd": "PhD",
}

df_clean["gender"] = (
    df_clean["gender"].str.strip().str.lower().map(gender_map)
)
df_clean["education"] = (
    df_clean["education"].str.strip().str.lower().map(education_map)
)

# These values are fairly impossible rather than merely unusual. 
df_clean.loc[df_clean["age"] > 110, "age"] = np.nan
df_clean.loc[df_clean["annual_income_eur"] < 0, "annual_income_eur"] = np.nan

print("Rows after removing duplicates:", len(df_clean))
print("\nStandardised gender labels:")
print(df_clean["gender"].value_counts(dropna=False))

Rows after removing duplicates: 600

Standardised gender labels:
gender
Male                 275
Female               260
NaN                   30
Non-binary            21
Prefer not to say     14
Name: count, dtype: int64


**Checkpoint.** Run this cell. If an assertion fails, go back and fix section 2.

In [ ]:
assert df_clean.duplicated().sum() == 0, "There are still duplicate rows"
assert set(df_clean["gender"].dropna()) <= {"Female", "Male", "Non-binary", "Prefer not to say"}, "Gender labels not standardised"
assert set(df_clean["education"].dropna()) == {"High School", "Bachelor", "Master", "PhD"}, "Education labels not standardised"
assert df_clean["age"].max() <= 110, "Impossible ages remain"
assert (df_clean["annual_income_eur"].dropna() >= 0).all(), "Negative incomes remain"
print("Section 2 checks passed:", df_clean.shape)

Section 2 checks passed: (600, 11)


**Answer — "Think first"**

- **Fixed automatically:** exact duplicates and spelling variants of the same label (`"F"` → `"Female"`). There is only one correct answer, so a fixed rule can apply it.
- **Needs a decision:** where "impossible" begins (why 110 years, and not 100?), whether a value is an error or just unusual, and what to do with labels that don't match any known category. In `gender_map`, any unmatched label becomes `NaN`, and that is itself a decision.
- These steps are allowed **before** the split because each row is fixed on its own. No statistic is calculated from the other rows.

---
## 3. Target and features — check for target leakage

We want to predict `satisfaction_score`.

**Think:** for every column, ask *would this be available **before** we know the satisfaction score?*

- Which column only exists **because of** the outcome? That is **target leakage**: remove that column.
- Which column carries no information at all and should not be a feature?

In [ ]:
target = "satisfaction_score"

# A respondent ID is only an identifier. A complaint filed after the survey
# is known too late to use when predicting satisfaction at survey time.
leaky_cols = ["complaint_filed_after_survey"]
id_cols = ["respondent_id"]

X = df_clean.drop(columns=[target] + leaky_cols + id_cols)
y = df_clean[target]
print("Features used:", X.columns.tolist())

Features used: ['age', 'gender', 'education', 'household_size', 'annual_income_eur', 'survey_date', 'hours_online_per_week', 'online_purchases_last_month']


**Answer**

- `complaint_filed_after_survey` is **target leakage**. A complaint is filed *after* the survey, often *because* the person was unhappy, so it won't exist yet when we want to predict satisfaction.
- `respondent_id` carries **no information**. It's an arbitrary label, so a model could only memorise it, which doesn't generalise to new people.

---
## 4. Train / test split

From here on, **anything that learns from the data is fitted on `X_train` only** and then applied to `X_test`.

The test set is looked at **once, at the end**.

In [ ]:
# Keep 20% of the rows aside. random_state makes this split reproducible.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("train:", X_train.shape, " test:", X_test.shape)

# Sections 5-10 change Xtr/Xte. These copies keep X_train/X_test untouched
# for the all-in-one pipeline in section 11.
Xtr, Xte = X_train.copy(), X_test.copy()

train: (480, 8)  test: (120, 8)


---
## 5. Missing values

Missing ≠ zero ≠ unknown ≠ not applicable.

**Observe:** how much is missing in each column of `Xtr`? Some of these `NaN`s are values you created in section 2.

**Think:** for each column, which option fits — median, mean, most frequent, an `"Unknown"` category, or dropping rows?
- Why is the median usually safer than the mean for income?
- Why might `"Unknown"` be fine for `gender` but awkward for `education` (an ordinal variable)?

**Code:** compute the fill values **from `Xtr`**, then apply the *same* values to both `Xtr` and `Xte`.

In [ ]:
# These are the columns where a median can be appropriate.
numeric_cols = [
    "age", "household_size", "annual_income_eur", "hours_online_per_week",
    "online_purchases_last_month",
]

print("Missing values in the training data:")
print(Xtr.isna().sum().sort_values(ascending=False))

# Learn one median per numeric column from TRAINING data only.
# The test set must not influence this choice.
medians = Xtr[numeric_cols].median()

# Reuse the training medians in both datasets.
Xtr[numeric_cols] = Xtr[numeric_cols].fillna(medians)
Xte[numeric_cols] = Xte[numeric_cols].fillna(medians)

# "Unknown" is a meaningful extra category for a nominal variable such as gender.
Xtr["gender"] = Xtr["gender"].fillna("Unknown")
Xte["gender"] = Xte["gender"].fillna("Unknown")

# Education is ordinal, so we use the most common TRAINING value instead.
education_mode = Xtr["education"].mode()[0]
Xtr["education"] = Xtr["education"].fillna(education_mode)
Xte["education"] = Xte["education"].fillna(education_mode)

print("\nMissing values after imputation:")
print(Xtr.isna().sum().sum(), "in train;", Xte.isna().sum().sum(), "in test")

Missing values in the training data:
annual_income_eur              40
age                            27
gender                         24
education                      19
household_size                 14
survey_date                     0
hours_online_per_week           0
online_purchases_last_month     0
dtype: int64

Missing values after imputation:
0 in train; 0 in test


**Answer**

- **Median vs mean for income:** income is right-skewed, with a few very large values (up to €480,000). Those pull the **mean** upwards, while the **median** (the middle value) barely moves. That makes the median a more typical fill value.
- **`"Unknown"` for gender:** gender is **nominal**, with no order between categories. `"Unknown"` simply becomes one more category, and "didn't answer" may even carry some meaning.
- **Why `"Unknown"` is awkward for education:** education is **ordinal** (High School < Bachelor < Master < PhD). `"Unknown"` has no place in that order, and the `OrdinalEncoder` in Section 9 would have no number to give it. Filling with the training **mode** keeps the variable ordered.
- **Training values only:** `medians` and `education_mode` are calculated from `Xtr`, then reused for `Xte`.

---
## 6. Outliers and suspicious values

The *impossible* values were handled in section 2. What is left may be **unusual but real**.

**Observe:** use descriptive statistics and at least one plot to investigate `age` and `annual_income_eur` in `Xtr`.

**Think:** is an unusual value necessarily an error? Would you remove an income of €480,000? What would removing it do to the model — and to the people it describes?

In [ ]:
print(Xtr[["age", "annual_income_eur"]].describe())
print("\nFive largest incomes:")
print(Xtr["annual_income_eur"].nlargest(5))

Xtr[["age", "annual_income_eur"]].hist(bins=20, figsize=(10, 3))
plt.tight_layout()
plt.show()

# Minimal question: Based on the output, should the largest income be kept?
# It is unusual but still plausible, so we keep it.

              age  annual_income_eur
count  480.000000         480.000000
mean    41.793750       58348.541667
std     11.962762       36589.444243
min     18.000000       15200.000000
25%     34.000000       37800.000000
50%     42.000000       51200.000000
75%     49.000000       68100.000000
max     80.000000      480000.000000

Five largest incomes:
131    480000.0
149    310000.0
289    238700.0
593    180600.0
605    167800.0
Name: annual_income_eur, dtype: float64


<Figure size 1000x300 with 2 Axes>

**Answer**

- The age distribution looks reasonable (18–80) once the impossible ages are gone.
- Income is strongly right-skewed. The median is about €51k, but the maximum is €480k.
- **An unusual value is not necessarily an error.** €480,000 is rare but possible for a real person, so **we keep it**. Deleting it would quietly remove high earners from the data, and the model would then perform worse for exactly those people. Better options are a robust treatment, such as a log transform of income or a robust scaler, or simply documenting the decision.
- Only values that are **impossible** (handled in Section 2) are treated as errors.

---
## 7. Dates

`survey_date` is stored as text. Convert it to a datetime and extract at least one useful feature.

**Think:** some dates can't be parsed. What should happen to them? Is this step row-wise (no learning) or not?

In [ ]:
def add_date_features(data):
    # Work on a copy so this function does not modify its input dataframe.
    result = data.copy()

    # Invalid date strings become NaT rather than causing an error.
    dates = pd.to_datetime(result["survey_date"], errors="coerce")

    # A date contains useful calendar information for a later model.
    result["survey_month"] = dates.dt.month
    result["survey_dayofweek"] = dates.dt.dayofweek

    # We have extracted the useful parts, so the original text column is no longer needed.
    return result.drop(columns="survey_date")

# Apply exactly the same row-wise operation to train and test data.
Xtr = add_date_features(Xtr)
Xte = add_date_features(Xte)

# If parsing made a date part missing, learn replacement values from training data.
date_cols = ["survey_month", "survey_dayofweek"]
date_medians = Xtr[date_cols].median()
Xtr[date_cols] = Xtr[date_cols].fillna(date_medians)
Xte[date_cols] = Xte[date_cols].fillna(date_medians)
Xtr[["survey_month", "survey_dayofweek"]].head()

,survey_month,survey_dayofweek
145,12.0,6.0
9,3.0,4.0
377,4.0,4.0
532,3.0,0.0
188,2.0,6.0


**Answer:** Dates that cannot be parsed become missing values (`NaT`), so the code fills their month and day-of-week values using medians learned from the training set. Extracting the date parts is row-wise because each date is handled independently; calculating the replacement medians is the part that learns from the training data.

---
## 8. Feature engineering

Create **at least two new features** that could plausibly help predict satisfaction. Write one sentence per feature explaining why.

Ideas: income per household member, age group, purchases per online hour.

**Why here, before encoding and scaling?** A ratio of two *scaled* variables is meaningless — engineer features from the original units.

In [ ]:
def add_features(data):
    # Work on a copy, so the same function is safe for either split.
    result = data.copy()

    # A household with the same income may have a different situation depending on its size.
    # clip(lower=1) prevents division by zero.
    result["income_per_person"] = (
        result["annual_income_eur"] / result["household_size"].clip(lower=1)
    )

    # clip(lower=0.1) prevents a zero number of online hours from causing an error.
    result["purchases_per_online_hour"] = (
        result["online_purchases_last_month"] / result["hours_online_per_week"].clip(lower=0.1)
    )

    # Age groups are categories, not numeric scores; they will be one-hot encoded later.
    result["age_group"] = pd.cut(
        result["age"], bins=[0, 29, 44, 64, np.inf],
        labels=["18-29", "30-44", "45-64", "65+"],
    ).astype("object")
    return result

# The recipe is fixed, so apply it identically to train and test data.
Xtr = add_features(Xtr)
Xte = add_features(Xte)
Xtr[["income_per_person", "purchases_per_online_hour", "age_group"]].head()

,income_per_person,purchases_per_online_hour,age_group
145,23966.666667,0.392157,18-29
9,16150.000000,0.166667,45-64
377,114500.000000,0.442478,45-64
532,51200.000000,0.089286,30-44
188,14250.000000,0.000000,45-64


**Answer — one sentence per feature**

1. **`income_per_person`**: the same income supports a single person very differently from a family of five, so income per household member is a better measure of financial comfort, which may affect satisfaction.
2. **`purchases_per_online_hour`**: this measures how purchase-driven someone's time online is, as opposed to simply how much time they spend online, and could reflect how engaged they are as a customer.
3. **`age_group`**: satisfaction may not change in a straight line with age, and life-stage groups can capture effects that a single number misses.

**Tutorial:** These features are calculated **before scaling**, while values are still in euros, hours and counts. A ratio of two standardised values has no real-world meaning.

---
## 9. Encoding categorical variables

**Think:** for `education`, `gender` and `age_group` — nominal or ordinal?

- **Ordinal** → `OrdinalEncoder` with the categories in an **explicit order**. (Don't use `LabelEncoder` — it sorts alphabetically and is meant for targets.)
- **Nominal** → `OneHotEncoder(handle_unknown="ignore")`, so a category that only appears in the test set doesn't crash the model.

Fit the encoders on `Xtr`, transform both.

In [ ]:
# This order is meaningful, so education is an ordinal variable.
edu_order = ["High School", "Bachelor", "Master", "PhD"]
education_encoder = OrdinalEncoder(categories=[edu_order])

# fit_transform learns the order from the training configuration and encodes training rows.
Xtr[["education"]] = education_encoder.fit_transform(Xtr[["education"]])

# transform applies exactly the same mapping to the test rows.
Xte[["education"]] = education_encoder.transform(Xte[["education"]])

# Gender and age group have no meaningful numeric order, so create separate 0/1 columns.
nominal_cols = ["gender", "age_group"]
nominal_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# Learn which category columns exist from training data only.
train_nominal = pd.DataFrame(
    nominal_encoder.fit_transform(Xtr[nominal_cols]),
    columns=nominal_encoder.get_feature_names_out(nominal_cols), index=Xtr.index,
)

# An unseen test category is ignored rather than causing an error.
test_nominal = pd.DataFrame(
    nominal_encoder.transform(Xte[nominal_cols]),
    columns=nominal_encoder.get_feature_names_out(nominal_cols), index=Xte.index,
)

# Replace the original text columns with their new numeric columns.
Xtr = pd.concat([Xtr.drop(columns=nominal_cols), train_nominal], axis=1)
Xte = pd.concat([Xte.drop(columns=nominal_cols), test_nominal], axis=1)
print("Columns after encoding:", Xtr.shape[1])

Columns after encoding: 19


**Answer — nominal or ordinal?**

| Column | Type | Encoder | Why |
|---|---|---|---|
| `education` | ordinal | `OrdinalEncoder` with an explicit order | High School < Bachelor < Master < PhD has a real order |
| `gender` | nominal | `OneHotEncoder` | no category is "more" than another |
| `age_group` | ordinal in principle | `OneHotEncoder` here | the groups are ordered, but one-hot lets the model learn a separate effect for each group instead of assuming equal steps. Either choice can be defended. |

The **19 columns** after encoding are: 10 numeric columns (including `education` and the date and engineered features), 5 gender columns (4 labels + `Unknown`) and 4 age-group columns. If your count is different, check which categories appear in **train**.

---
## 10. Scaling numerical variables

Some columns use very different units. For example, age might be around 20-80, while annual income can be tens of thousands. A later algorithm could treat income as more important simply because its numbers are larger.

**Standardisation** changes each numeric column so that its training values are centred around 0 and use a comparable scale. It does not change which values are relatively high or low within a column.

The code below:

1. compares the original ranges;
2. learns a mean and standard deviation from the training data; and
3. uses those same training values to standardise both the training and test sets.

The one-hot columns already contain only 0 and 1, so we leave them unchanged.

In [ ]:
# One-hot columns are already 0/1, so scale only continuous or count-based variables.
numeric_cols = [
    "age", "household_size", "annual_income_eur", "hours_online_per_week",
    "online_purchases_last_month", "survey_month", "survey_dayofweek",
    "income_per_person", "purchases_per_online_hour",
]
print("Ranges before scaling:")
print(Xtr[numeric_cols].agg(["min", "max"]))

# The scaler learns the mean and standard deviation from training data only.
scaler = StandardScaler()
Xtr[numeric_cols] = scaler.fit_transform(Xtr[numeric_cols])

# The test set uses those same training values; it must not calculate new ones.
Xte[numeric_cols] = scaler.transform(Xte[numeric_cols])

print("\nTraining means after scaling:")
print(Xtr[numeric_cols].mean().round(2))

Ranges before scaling:
      age  household_size  annual_income_eur  hours_online_per_week  \
min  18.0             1.0            15200.0                    0.0   
max  80.0             6.0           480000.0                   61.8   

     online_purchases_last_month  survey_month  survey_dayofweek  \
min                          0.0           1.0               0.0   
max                         20.0          12.0               6.0   

     income_per_person  purchases_per_online_hour  
min             3400.0                   0.000000  
max           480000.0                   1.666667  

Training means after scaling:
age                           -0.0
household_size                -0.0
annual_income_eur              0.0
hours_online_per_week         -0.0
online_purchases_last_month    0.0
survey_month                  -0.0
survey_dayofweek              -0.0
income_per_person              0.0
purchases_per_online_hour      0.0
dtype: float64


**Answer:** The training columns have means close to 0 because the scaler used their own means and standard deviations. The test columns do not need to have means of 0: they use the training values so that the test data stays genuinely unseen.

---
## 11. Doing the same preparation with a pipeline

So far, we have prepared the data one step at a time: fill missing values, extract date information, create new features, encode categories, and scale numeric columns.

A **pipeline** is simply a saved checklist of those steps. Instead of writing and running each step separately every time, we put the steps in the correct order once and ask Python to run that checklist for us.

For this notebook, the pipeline does the following:

1. Starts with the original training or test table.
2. Extracts month and day of week from `survey_date` and creates the two ratio features.
3. Fills missing numeric values, then standardises numeric columns.
4. Fills and encodes `education` in its meaningful order.
5. Fills and turns `gender` and `age_group` into 0/1 columns.

When we run `fit_transform(X_train)`, the pipeline learns any required values, such as medians and category lists, from the training data and prepares it. When we run `transform(X_test)`, it follows the exact same checklist for the test data but reuses the values learned from training.

This keeps our preparation consistent and prevents us from accidentally learning from the test data. It is still only data preparation; we are not building a predictive model yet.

In [ ]:
def row_wise_features(data):
    """Create features using each row only; no dataset-wide values are learned here."""
    result = data.copy()

    # Turn one date column into two numeric calendar features.
    dates = pd.to_datetime(result["survey_date"], errors="coerce")
    result["survey_month"] = dates.dt.month
    result["survey_dayofweek"] = dates.dt.dayofweek

    # Create ratios before scaling, while values still use their original units.
    result["income_per_person"] = (
        result["annual_income_eur"] / result["household_size"].clip(lower=1)
    )
    result["purchases_per_online_hour"] = (
        result["online_purchases_last_month"] / result["hours_online_per_week"].clip(lower=0.1)
    )

    # Keep age bands as categories; the nominal pipeline will encode them.
    result["age_group"] = pd.cut(
        result["age"], bins=[0, 29, 44, 64, np.inf],
        labels=["18-29", "30-44", "45-64", "65+"],
    ).astype("object")
    return result.drop(columns="survey_date")

# Assign each column to the type of preprocessing it needs.
numeric_cols = [
    "age", "household_size", "annual_income_eur", "hours_online_per_week",
    "online_purchases_last_month", "survey_month", "survey_dayofweek",
    "income_per_person", "purchases_per_online_hour",
]
ordinal_cols = ["education"]
nominal_cols = ["gender", "age_group"]
edu_order = ["High School", "Bachelor", "Master", "PhD"]

# For numeric data: fill gaps with training medians, then put columns on a common scale.
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

# For education: fill gaps, then encode the supplied meaningful order.
ordinal_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OrdinalEncoder(categories=[edu_order])),
])

# For nominal data: label missing values and create a 0/1 column for each category.
nominal_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])

# Send each group of columns through its appropriate mini-pipeline.
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("ord", ordinal_pipeline, ordinal_cols),
    ("nom", nominal_pipeline, nominal_cols),
])

# First create features, then preprocess the resulting columns.
preprocessing_pipeline = Pipeline([
    ("features", FunctionTransformer(row_wise_features)),
    ("preprocess", preprocessor),
])

# fit_transform learns every required value from the training data and transforms it.
X_train_prepared = preprocessing_pipeline.fit_transform(X_train)

# transform applies those already-learned values to the unseen test data.
X_test_prepared = preprocessing_pipeline.transform(X_test)
print("Prepared train shape:", X_train_prepared.shape)
print("Prepared test shape: ", X_test_prepared.shape)

Prepared train shape: (480, 20)
Prepared test shape:  (120, 20)


### Check your preprocessing output

After completing the scaffold, inspect the transformed arrays or dataframe:

- How many rows and columns are there now?
- Are there any missing values left?
- Which original categorical columns became several binary columns?
- Which values were learned from `X_train`, rather than from the full dataset?

There is no model training or model score in this notebook; those belong to Week 5.

In [ ]:
# Inspect the pipeline output.
# The pipeline returns a NumPy array, so turn it back into a labelled DataFrame first.
feature_names = preprocessing_pipeline.named_steps["preprocess"].get_feature_names_out()
train_prepared_df = pd.DataFrame(
    X_train_prepared.toarray() if hasattr(X_train_prepared, "toarray") else X_train_prepared,
    columns=feature_names, index=X_train.index,
)

print("Rows, columns:", train_prepared_df.shape)
print("Missing values left:", int(train_prepared_df.isna().sum().sum()))
print("\nColumns created from categorical variables:")
print([c for c in feature_names if c.startswith("nom__")])

# Values learned from X_train (examples)
num_step = preprocessing_pipeline.named_steps["preprocess"].named_transformers_["num"]
print("\nMedians learned from train:", dict(zip(numeric_cols, num_step.named_steps["impute"].statistics_.round(1))))
print("Means learned by the scaler:", dict(zip(numeric_cols, num_step.named_steps["scale"].mean_.round(1))))

**Answer**

- **Shape:** 480 rows × 20 columns for train, and 120 × 20 for test. There is one more column than the manual version's 19. Gender has 5 categories in both versions, but in the pipeline `age_group` is created before imputation. An age that was missing therefore gives a missing age group, which becomes an extra `age_group_Unknown` column. In the manual version, age was imputed first.
- **Missing values left:** 0.
- **Categorical columns that became several 0/1 columns:** `gender` and `age_group`. `education` stays as a single ordinal column.
- **Learned from `X_train` only:** the numeric medians (`SimpleImputer`), the most frequent education level, the list of categories for each one-hot column, and the means and standard deviations (`StandardScaler`). `transform(X_test)` reuses all of these without recalculating them.

---
## 12. Leakage, revisited

### 12a. Preprocessing leakage

**Question:** should imputation medians and scaling parameters be calculated on the entire dataset before splitting?

Use the output below to compare the values. Then explain in 2–3 sentences why the training-only version is the correct workflow.

The difference may be small in this dataset. The important issue is whether the test set influenced a decision before evaluation.

### 12b. Target leakage

`complaint_filed_after_survey` is recorded after the survey. Should it be available as an input when predicting satisfaction at survey time?

Explain:

1. When is this column known?
2. When would the prediction be made?
3. Why can a train/test split fail to reveal this kind of leakage?

In [ ]:
col = "annual_income_eur"
print(f"median from ALL data   : {X[col].median():>10,.0f}")
print(f"median from TRAIN only : {X_train[col].median():>10,.0f}")
print(f"mean   from ALL data   : {X[col].mean():>10,.0f}")
print(f"mean   from TRAIN only : {X_train[col].mean():>10,.0f}")

median from ALL data   :     51,300
median from TRAIN only :     51,200
mean   from ALL data   :     58,834
mean   from TRAIN only :     58,998


**Answer 12a — preprocessing leakage**

No, they shouldn't. The test set stands in for **future, unseen data**. If medians, means or standard deviations are calculated on the full dataset, information from the test rows enters the preprocessing, and the test score becomes slightly too optimistic. Here the difference is small (median €51,300 vs €51,200), but the principle matters: the test set must not influence any decision made before evaluation. Learning parameters on train only, then reusing them, mirrors what happens in real life, where new data arrives after the model has been built.

**Answer 12b — target leakage**

1. **When is it known?** Only *after* the survey, sometimes weeks later, once the customer has decided to complain.
2. **When is the prediction made?** At survey time, before any complaint exists.
3. **Why doesn't a train/test split catch it?** The column is present in **both** halves of the historical data, so the model would score well on the test set too. The split is random over rows, not over time. The problem only shows up in practice, when the column is always empty at prediction time and the model's performance collapses.

A train/test split cannot reliably catch target leakage: the leaked column can appear in both splits. The problem appears when the system is used in practice, because at prediction time the complaint has not been filed yet.

*Always ask: would this feature be available before the outcome is known?*

---
# Exit ticket

1. Why are the median, category mappings, and scaling values learned from `X_train` rather than the full dataset?
2. Why is `complaint_filed_after_survey` target leakage when predicting `satisfaction_score` at survey time?

**Exit ticket answers**

1. The test set has to behave like genuinely **new, unseen data**. If the median, category mappings or scaling values were learned from the full dataset, the test rows would have influenced the preprocessing, and the evaluation would be too optimistic. Learning them from `X_train` and reusing them on `X_test` copies what happens when the model is used on new people.
2. The complaint is recorded **after** the survey, often **as a result of** low satisfaction. At the moment we want to predict satisfaction it doesn't exist yet, so a model that relies on it would look excellent in testing but couldn't be used in practice.

---
---
# Part B — The same workflow on `dataset.csv` (student social-media data)

This part repeats the 12 steps on a new dataset: 5,000 students, their social-media habits, sleep, study time and a `Mental_Health_Score` (roughly 1–10).

**Goal:** prepare the data to predict `Mental_Health_Score`. As in Part A, there is no model yet; that comes in Week 5.

Variables in Part B start with `st_`, so they don't overwrite anything from Part A.

In [ ]:
# Change this path if dataset.csv is somewhere else in your repo (e.g. "../data/dataset.csv").
DATA_PATH = "dataset.csv"

students = pd.read_csv(DATA_PATH)
students.head()

## B1. Inspect
Same three checks as Part A: shape and types, then missing values, duplicates and summary statistics, then category counts.

In [ ]:
print("Rows, columns:", students.shape)
students.info()

In [ ]:
print("Missing values:", int(students.isna().sum().sum()))
print("Exact duplicate rows:", students.duplicated().sum())
students.describe().round(2)

In [ ]:
st_categorical = ["Gender", "Country", "Academic_Level", "Most_Used_Platform",
                  "Purpose_Of_Use", "Stress_Level"]
for col in st_categorical:
    counts = students[col].value_counts(dropna=False)
    print(f"--- {col} ({counts.size} categories) ---")
    print(counts.head(8), "\n")

**What the output shows**

- **5,000 rows × 13 columns** with **no missing values**. This dataset is much cleaner than the survey.
- **2 exact duplicate rows.**
- **Read the min and max:** `Physical_Activity_Hours` has a **minimum of −0.4**, which is impossible because nobody exercises for negative hours. Everything else is plausible: ages 18–24, sleep 3.6–9.9 h.
- `Country` has **over 100 categories**, many with only a handful of students. This matters for encoding in B9.
- The labels are already consistent (`Male`/`Female`, three academic levels).

## B2. Row-wise cleaning (before the split)
1. Drop exact duplicates.
2. Strip stray spaces from text labels, as a safety rule. There are no spelling variants here.
3. Set **impossible** values to `NaN`: negative hours, or more than 24 hours in a day.

In [ ]:
st_clean = students.drop_duplicates().copy()

for col in st_categorical:
    st_clean[col] = st_clean[col].str.strip()

hour_cols = ["Avg_Daily_Usage_Hours", "Study_Hours",
             "Physical_Activity_Hours", "Sleep_Hours_Per_Night"]
for col in hour_cols:
    impossible = (st_clean[col] < 0) | (st_clean[col] > 24)
    print(f"{col}: {impossible.sum()} impossible values set to NaN")
    st_clean.loc[impossible, col] = np.nan

# Sanity check across columns: total hours per day can't exceed 24.
total_hours = st_clean[hour_cols].sum(axis=1)
print("\nMaximum total hours in one day:", total_hours.max())
print("Rows after removing duplicates:", len(st_clean))

In [ ]:
# Checkpoint
assert st_clean.duplicated().sum() == 0, "Duplicates remain"
assert (st_clean[hour_cols].dropna() >= 0).all().all(), "Negative hours remain"
assert (st_clean[hour_cols].dropna() <= 24).all().all(), "More than 24 hours remain"
print("B2 checks passed:", st_clean.shape)

## B3. Target and features — check for target leakage

Target: `Mental_Health_Score`.

**Think:** is `Stress_Level` a *cause* we could know in advance, or **another measurement of the same outcome**, recorded at the same moment?

**Decision:** we **drop `Stress_Level`** from the features. It's self-reported in the same questionnaire as the target and measures closely related wellbeing (average score is 7.8 for *Low* stress and 5.1 for *Very High*). If we want to predict mental health from **behaviour**, stress is part of the outcome, not an input. A model using it would mostly be learning "unhappy people report being unhappy".

*This is a judgement call. If stress were measured earlier, for example at the start of term, it could be a legitimate feature. Write down whichever choice you make and why.*

There's no ID column to remove.

In [ ]:
st_target = "Mental_Health_Score"
st_leaky = ["Stress_Level"]

print(students.groupby("Stress_Level")[st_target].mean().round(2).sort_values())

st_X = st_clean.drop(columns=[st_target] + st_leaky)
st_y = st_clean[st_target]
print("\nFeatures used:", st_X.columns.tolist())

## B4. Train / test split

In [ ]:
st_X_train, st_X_test, st_y_train, st_y_test = train_test_split(
    st_X, st_y, test_size=0.2, random_state=42
)
print("train:", st_X_train.shape, " test:", st_X_test.shape)

# Working copies for the manual steps B5–B10; the originals are kept for the pipeline in B11.
sXtr, sXte = st_X_train.copy(), st_X_test.copy()

## B5. Missing values
The only gaps are the ones **we created** in B2 (negative activity hours). Physical activity is a skewed count of hours, so we use the **training median**.

In [ ]:
st_numeric = ["Age", "Avg_Daily_Usage_Hours", "Daily_Unlocks", "Study_Hours",
              "Physical_Activity_Hours", "Sleep_Hours_Per_Night"]

print("Missing in train before:")
print(sXtr.isna().sum()[sXtr.isna().sum() > 0])

st_medians = sXtr[st_numeric].median()          # learned from TRAIN only
sXtr[st_numeric] = sXtr[st_numeric].fillna(st_medians)
sXte[st_numeric] = sXte[st_numeric].fillna(st_medians)

print("\nMissing after:", int(sXtr.isna().sum().sum()), "train /", int(sXte.isna().sum().sum()), "test")

## B6. Outliers
We use the IQR rule (values beyond 1.5 × IQR from the quartiles) to *flag* unusual values. Flagged values are investigated, not automatically deleted.

In [ ]:
q1, q3 = sXtr[st_numeric].quantile(0.25), sXtr[st_numeric].quantile(0.75)
iqr = q3 - q1
flagged = ((sXtr[st_numeric] < q1 - 1.5 * iqr) | (sXtr[st_numeric] > q3 + 1.5 * iqr)).sum()
print("Values flagged by the IQR rule (train):")
print(flagged)

sXtr[["Avg_Daily_Usage_Hours", "Daily_Unlocks", "Sleep_Hours_Per_Night"]].hist(bins=25, figsize=(12, 3), layout=(1, 3))
plt.tight_layout()
plt.show()

**Answer:** Only 12 values are flagged: 2 in `Study_Hours` and 10 in `Physical_Activity_Hours`, at the high end (about 8 hours of study, or 4 hours of exercise a day). Social-media use and unlocks have no flags at all; even 9 hours a day or 270 unlocks sits within the normal spread. All of these are plausible for real students, not errors, so **we keep them**. Removing them would make the model worse for the most and least active students.

## B7. Dates
`dataset.csv` has **no date column**, so this step doesn't apply. In real projects, check whether "no date" means the data really is a single snapshot. If it is, the model can't learn anything about change over time.

## B8. Feature engineering
Three new features, created from the original units before encoding and scaling:

1. **`social_to_study_ratio`** — social-media hours per study hour. The *balance* between the two may matter more than either one on its own.
2. **`unlocks_per_usage_hour`** — how often the phone is picked up per hour of use. Many short, compulsive checks are different from a few long sessions.
3. **`sleep_deficit`** — hours below the recommended 7 hours of sleep (0 if 7 or more). This captures the idea that lack of sleep hurts, while extra sleep beyond 7 hours doesn't keep helping.

In [ ]:
def st_add_features(data):
    result = data.copy()
    result["social_to_study_ratio"] = (
        result["Avg_Daily_Usage_Hours"] / result["Study_Hours"].clip(lower=0.1)
    )
    result["unlocks_per_usage_hour"] = (
        result["Daily_Unlocks"] / result["Avg_Daily_Usage_Hours"].clip(lower=0.1)
    )
    result["sleep_deficit"] = (7 - result["Sleep_Hours_Per_Night"]).clip(lower=0)
    return result

sXtr = st_add_features(sXtr)
sXte = st_add_features(sXte)

st_new = ["social_to_study_ratio", "unlocks_per_usage_hour", "sleep_deficit"]
# Quick check: how does each new feature relate to the target in TRAIN?
print(sXtr[st_new].corrwith(st_y_train).round(2))
sXtr[st_new].head()

## B9. Encoding
| Column | Type | Encoder |
|---|---|---|
| `Academic_Level` | ordinal (High School < Undergraduate < Graduate) | `OrdinalEncoder`, explicit order |
| `Gender`, `Most_Used_Platform`, `Purpose_Of_Use` | nominal | `OneHotEncoder` |
| `Country` | nominal with **100+ categories** | `OneHotEncoder(min_frequency=20)`: rare countries are grouped into one *infrequent* column |

**Why `min_frequency`?** Without it, `Country` alone would create over 100 columns, most of them almost always 0. A country seen only twice in training can't teach the model anything reliable.

In [ ]:
st_level_order = ["High School", "Undergraduate", "Graduate"]
st_ordinal_enc = OrdinalEncoder(categories=[st_level_order])
sXtr[["Academic_Level"]] = st_ordinal_enc.fit_transform(sXtr[["Academic_Level"]])
sXte[["Academic_Level"]] = st_ordinal_enc.transform(sXte[["Academic_Level"]])

st_nominal = ["Gender", "Country", "Most_Used_Platform", "Purpose_Of_Use"]
st_onehot = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20,
                          sparse_output=False)
tr_nom = pd.DataFrame(st_onehot.fit_transform(sXtr[st_nominal]),
                      columns=st_onehot.get_feature_names_out(st_nominal), index=sXtr.index)
te_nom = pd.DataFrame(st_onehot.transform(sXte[st_nominal]),
                      columns=st_onehot.get_feature_names_out(st_nominal), index=sXte.index)

sXtr = pd.concat([sXtr.drop(columns=st_nominal), tr_nom], axis=1)
sXte = pd.concat([sXte.drop(columns=st_nominal), te_nom], axis=1)

print("Columns after encoding:", sXtr.shape[1])
print("Country columns:", [c for c in tr_nom.columns if c.startswith("Country")])

## B10. Scaling

In [ ]:
st_scale_cols = st_numeric + st_new
st_scaler = StandardScaler()
sXtr[st_scale_cols] = st_scaler.fit_transform(sXtr[st_scale_cols])   # learn on train
sXte[st_scale_cols] = st_scaler.transform(sXte[st_scale_cols])       # reuse on test

print("Train means (should be ~0):", sXtr[st_scale_cols].mean().round(2).abs().max())
print("Test means (close to 0, but not exactly):")
print(sXte[st_scale_cols].mean().round(2))

## B11. The whole thing as one pipeline
The same recipe as Part A: row-wise features first, then a `ColumnTransformer` that sends each group of columns to its own mini-pipeline. The imputers stay in the pipeline even though this dataset has almost no gaps, because **new data might**.

In [ ]:
st_numeric_all = st_numeric + st_new

st_preprocessor = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), st_numeric_all),
    ("ord", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("encode", OrdinalEncoder(categories=[st_level_order]))]), ["Academic_Level"]),
    ("nom", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
                      ("encode", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                               min_frequency=20, sparse_output=False))]), st_nominal),
])

st_pipeline = Pipeline([
    ("features", FunctionTransformer(st_add_features)),
    ("preprocess", st_preprocessor),
])

st_train_prepared = st_pipeline.fit_transform(st_X_train)   # learn from train
st_test_prepared = st_pipeline.transform(st_X_test)         # reuse on test

print("Prepared train:", st_train_prepared.shape, "| test:", st_test_prepared.shape)
print("Missing values left:", int(np.isnan(st_train_prepared).sum()))
print("Same result as the manual steps:", st_train_prepared.shape[1] == sXtr.shape[1])

## B12. Leakage, revisited

In [ ]:
col = "Daily_Unlocks"
print(f"mean from ALL data   : {st_X[col].mean():.2f}")
print(f"mean from TRAIN only : {st_X_train[col].mean():.2f}")
print(f"SD   from ALL data   : {st_X[col].std():.2f}")
print(f"SD   from TRAIN only : {st_X_train[col].std():.2f}")

**Answer**

- **Preprocessing leakage:** the differences are tiny, but the scaler and imputer still learn from **train only**, so the test set remains a fair stand-in for new students.
- **Target leakage:** the risky column here is `Stress_Level`, not an obvious "after the fact" column like the complaint in Part A. It is recorded at the same time and is strongly tied to the target, so it is really part of the outcome. A random split wouldn't flag it: the model would score very well, and the score would be misleading if the aim is to predict mental health from behaviour.

### Summary of decisions (Part B)
| Step | Decision |
|---|---|
| Duplicates | 2 removed |
| Impossible values | negative `Physical_Activity_Hours` → `NaN` → train median |
| Target leakage | `Stress_Level` dropped (co-measured outcome) |
| Outliers | flagged, investigated, **kept** (plausible heavy users) |
| Dates | none in this dataset |
| New features | `social_to_study_ratio`, `unlocks_per_usage_hour`, `sleep_deficit` |
| Encoding | ordinal `Academic_Level`; one-hot for the rest, rare countries grouped |
| Scaling | `StandardScaler`, fitted on train |